## 1. Import libraries and configure the inference environment

In [ ]:
from    typing      import  Literal, Sequence
from    itertools   import  product

from    tqdm.notebook   import  tqdm
from    time            import  perf_counter

import  torch
from    torch.utils.data                import  DataLoader

from    core.numerical.solver import  FastSM_Boltzmann_VHS, LAMBDA, one_step_RK4_classic
from    core.numerical.math   import  area_of_unit_sphere
from    core.utils            import  GaussianNormalizer, relative_error, sec_to_hms
from    core.neural.utils     import  count_parameters
from    core.neural.operator  import  ModifiedFNO, PFNO, ModifiedRFNO, PRFNO
import  matplotlib.pyplot               as  plt

from    loader      import  NormKeys, HistKeys
from    loader      import  path_base
from    loader      import  load_model_args, load_model_data
from    dataset     import  nonparametric_dataset, parametric_dataset
from    plot        import  *


DEVICE: torch.device = torch.device("cuda:1")

list_max_freq   = [6, 8, 10]
list_seed       = [0, 1, 2]
max_freq        = list_max_freq[1]
seed            = list_seed[0]

In [ ]:
DIMENSION   = 2
MAX_V       = 3/LAMBDA
VHS_COEFF   = 1.0/area_of_unit_sphere(DIMENSION)
min_t       = 0.0
max_t       = 5.0
delta_t     = 0.1


measured_method: Literal['fsm', 'fno', 'rfno', 'pfno', 'prfno'] = 'fno'

## 2. Instantiate the trained operators and normalizers

### 2-1. Load data (trained parameters, normalizers, and training histories)

In [ ]:
model_data_fno      = load_model_data("FNO2D-param", max_freq, seed)
model_data_pfno     = load_model_data("PFNO2D-param", max_freq, seed)
model_data_rfno     = load_model_data("RFNO2D-param", max_freq, seed)
model_data_prfno    = load_model_data("PRFNO2D-param", max_freq, seed)

### 2-2. Load trained operators

In [ ]:
model_fno   = ModifiedFNO(**load_model_args("FNO2D-param", max_freq))
model_pfno  = PFNO(**load_model_args("PFNO2D-param", max_freq))
model_rfno  = ModifiedRFNO(**load_model_args("RFNO2D-param", max_freq))
model_prfno = PRFNO(**load_model_args("PRFNO2D-param", max_freq))

model_fno.load_state_dict(  model_data_fno["state_dict"])
model_pfno.load_state_dict( model_data_pfno["state_dict"])
model_rfno.load_state_dict( model_data_rfno["state_dict"])
model_prfno.load_state_dict(model_data_prfno["state_dict"])
model_fno.to(DEVICE)
model_pfno.to(DEVICE)
model_rfno.to(DEVICE)
model_prfno.to(DEVICE)
model_fno.eval()
model_pfno.eval()
model_rfno.eval()
model_prfno.eval()
print(f"Successfully loaded models.")

### 2-4. Load normalizers

In [ ]:
norm_fno:   dict[NormKeys, GaussianNormalizer] = model_data_fno["normalizer"]
norm_pfno:  dict[NormKeys, GaussianNormalizer] = model_data_pfno["normalizer"]
norm_rfno:  dict[NormKeys, GaussianNormalizer] = model_data_rfno["normalizer"]
norm_prfno: dict[NormKeys, GaussianNormalizer] = model_data_prfno["normalizer"]
for v in norm_fno.values():   v.to(DEVICE)
for v in norm_pfno.values():  v.to(DEVICE)
for v in norm_rfno.values():  v.to(DEVICE)
for v in norm_prfno.values(): v.to(DEVICE)
print(f"Successfully loaded normalizers.")

## 3. Conduct inference

### 3-1. Construct dataloader

In [ ]:
path_data = path_base / "datasets" / "test" / "Boltzmann_2D__coeff1.59e-01__res_t051_v128"
assert path_data.exists(), f"The data path {path_data} does not exist."
list_alpha = [0.50]
part_start, part_end = 1, 1

data, params = [], []
generation_time = 0.0
files = sorted(path_data.glob("alpha*.pth"))
alpha_and_part = list(product(list_alpha, range(part_start, part_end+1)))
for alpha, part in tqdm(alpha_and_part):
    file = path_data / f"alpha{alpha:.2f}__part{part:03d}.pth"
    f = torch.load(file, weights_only=False)
    data.append(f['data'][[0]])         # (num_instances, res_t, 1, 1, res, res, 1)
    params.append(f['vhs_alpha']*torch.ones(1, 1))     # float
    generation_time += f['elapsed_time']
data    = torch.cat(data, dim=0).to(DEVICE)               # (num_total_instances, res_t, 1, 1, res, res, 1)
params  = torch.cat(params, dim=0).to(DEVICE)           # (num_total_instances, 1)
num_inst, res_t, res_v = data.size(0), data.size(1), data.size(-2)
data    = data.view(num_inst, res_t, res_v, res_v, 1)
data    = data.type(torch.float)
params  = params.type(torch.float)

print(f"Successfully loaded dataset with {num_inst} instances.")
print(f"Time for generating dataset\n>>> {sec_to_hms(int(generation_time))}")

In [ ]:
BATCH_SIZE = 2**6

dataloader_fno  = DataLoader(nonparametric_dataset(data, params, norm_fno), batch_size=BATCH_SIZE, shuffle=False)
dataloader_pfno = DataLoader(parametric_dataset(data, params, norm_pfno), batch_size=BATCH_SIZE, shuffle=False)
dataloader_rfno = DataLoader(nonparametric_dataset(data, params, norm_rfno), batch_size=BATCH_SIZE, shuffle=False)
dataloader_prfno = DataLoader(parametric_dataset(data, params, norm_prfno), batch_size=BATCH_SIZE, shuffle=False)
print(f"Successfully prepared dataloaders.")

### 3-2. Compute the trajectories on the test set

Fast spectral method

In [ ]:
if measured_method=='fsm':
    torch.cuda.reset_peak_memory_stats(DEVICE)
    torch.cuda.synchronize(DEVICE)

    fsm = FastSM_Boltzmann_VHS(DIMENSION, res_v, MAX_V, vhs_coeff=VHS_COEFF, vhs_alpha=alpha, dtype=torch.float32, device=DEVICE)
    arr_f = data[:, 0].reshape(1, *(1 for _ in range(DIMENSION)), *(res_v for _ in range(DIMENSION)), 1) # (batch, x, v, channel)

    inference_time_fsm = perf_counter()
    with torch.inference_mode():
        pred_fsm = fsm.solve(min_t, max_t, delta_t, arr_f, one_step_RK4_classic)
    inference_time_fsm = perf_counter() - inference_time_fsm
    pred_fsm = pred_fsm.reshape(num_inst, res_t, *(res_v for _ in range(DIMENSION)), 1).cpu()

    torch.cuda.synchronize(DEVICE)
    peak_memory_fsm = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)   # MB
    print(f"FSM Peak memory usage: {peak_memory_fsm:.2f}MB.")

FNO

In [ ]:
if measured_method=='fno':
    torch.cuda.reset_peak_memory_stats(DEVICE)
    torch.cuda.synchronize(DEVICE)
    pred_fno = []

    with torch.inference_mode():
        model_preds = []
        traj:   torch.Tensor
        alphas: torch.Tensor
        for traj, alphas in tqdm(dataloader_fno, desc="FNO Inference"):
            traj    = traj.to(DEVICE)
            alphas  = alphas.to(DEVICE)
            pred    = traj[:, 0]    # The initial condition of shape `(batch_size, res, res, 1)`
            # Save the initial condition
            batch_pred = [norm_fno['data'].decode(pred).cpu()]
            # Compute the trajectory
            for t in range(1, res_t):
                pred = model_fno.forward(pred, alphas)
                batch_pred.append(norm_fno['data'].decode(pred).cpu())
            batch_pred = torch.stack(batch_pred, dim=1)   # (batch_size, res_t, res, res, 1)
            model_preds.append(batch_pred)
        pred_fno = torch.cat(model_preds, dim=0)   # (num_inst, res_t, res, res, 1)

    torch.cuda.synchronize(DEVICE)
    peak_memory_fno = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)
    print(f"FNO Peak memory usage: {peak_memory_fno:.2f}MB.")

In [ ]:
if measured_method=='pfno':
    torch.cuda.reset_peak_memory_stats(DEVICE)
    torch.cuda.synchronize(DEVICE)
    pred_pfno = []

    with torch.inference_mode():
        model_preds = []
        traj:   torch.Tensor
        alphas: torch.Tensor
        for traj, alphas in tqdm(dataloader_pfno, desc="PFNO Inference"):
            traj    = traj.to(DEVICE)
            alphas  = alphas.to(DEVICE)
            pred    = traj[:, 0]    # The initial condition of shape `(batch_size, res, res, 1)`
            # Save the initial condition
            batch_pred = [norm_pfno['data'].decode(pred).cpu()]
            # Compute the trajectory
            for t in range(1, res_t):
                pred = model_pfno.forward(pred, alphas)
                batch_pred.append(norm_pfno['data'].decode(pred).cpu())
            batch_pred = torch.stack(batch_pred, dim=1)   # (batch_size, res_t, res, res, 1)
            model_preds.append(batch_pred)
        pred_pfno = torch.cat(model_preds, dim=0)   # (num_inst, res_t, res, res, 1)

    torch.cuda.synchronize(DEVICE)
    peak_memory_pfno = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)
    print(f"PFNO Peak memory usage: {peak_memory_pfno:.2f}MB.")

In [ ]:
if measured_method=='rfno':
    torch.cuda.reset_peak_memory_stats(DEVICE)
    torch.cuda.synchronize(DEVICE)
    pred_rfno = []

    with torch.inference_mode():
        model_preds = []
        traj:   torch.Tensor
        alphas: torch.Tensor
        for traj, alphas in tqdm(dataloader_rfno, desc="RFNO Inference"):
            traj    = traj.to(DEVICE)
            alphas  = alphas.to(DEVICE)
            pred    = traj[:, 0]    # The initial condition of shape `(batch_size, res, res, 1)`
            # Save the initial condition
            batch_pred = [norm_rfno['data'].decode(pred).cpu()]
            # Compute the trajectory
            for t in range(1, res_t):
                pred = model_rfno.forward(pred, alphas)
                batch_pred.append(norm_rfno['data'].decode(pred).cpu())
            batch_pred = torch.stack(batch_pred, dim=1)   # (batch_size, res_t, res, res, 1)
            model_preds.append(batch_pred)
        pred_rfno = torch.cat(model_preds, dim=0)   # (num_inst, res_t, res, res, 1)

    torch.cuda.synchronize(DEVICE)
    peak_memory_rfno = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)
    print(f"RFNO Peak memory usage: {peak_memory_rfno:.2f}MB.")

In [ ]:
if measured_method=='prfno':
    torch.cuda.reset_peak_memory_stats(DEVICE)
    torch.cuda.synchronize(DEVICE)
    pred_prfno = []
        
    with torch.inference_mode():
        model_preds = []
        traj:   torch.Tensor
        alphas: torch.Tensor
        for traj, alphas in tqdm(dataloader_prfno, desc="PRFNO Inference"):
            traj    = traj.to(DEVICE)
            alphas  = alphas.to(DEVICE)
            pred    = traj[:, 0]    # The initial condition of shape `(batch_size, res, res, 1)`
            # Save the initial condition
            batch_pred = [norm_prfno['data'].decode(pred).cpu()]
            # Compute the trajectory
            for t in range(1, res_t):
                pred = model_prfno.forward(pred, alphas)
                batch_pred.append(norm_prfno['data'].decode(pred).cpu())
            batch_pred = torch.stack(batch_pred, dim=1)   # (batch_size, res_t, res, res, 1)
            model_preds.append(batch_pred)
        pred_prfno = torch.cat(model_preds, dim=0)   # (num_inst, res_t, res, res, 1)
    
    torch.cuda.synchronize(DEVICE)
    peak_memory_prfno = torch.cuda.max_memory_allocated(DEVICE) / (1024**2)
    print(f"PRFNO Peak memory usage: {peak_memory_prfno:.2f}MB.")

End of file